# Java Code Summarization Dataset Building


## Data Mining Config Init.

Here I initialize the various directories to store cloned repos/mined java method/summary pairs and import required dependencies for dataset construction.

In [ ]:
import argparse
import base64
import csv
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from datasets import load_dataset
import subprocess
import shutil

import javalang
import requests
from lingua import Language, LanguageDetectorBuilder

PROJECT_ROOT = Path.cwd().parent
DEFAULT_DATA_DIR = PROJECT_ROOT / 'data'

@dataclass
class DatasetConfig:
    num_repos: int = 700 #num repos to clone
    min_stars: int = 100 #minimum stars for git api fetch
    max_search_pages: int = 10 #max number of pages of git api to search through when selecting repos
    max_files_per_repo: int = 40 #max java files used per repo
    train_size: int = 50000 #size of training set
    val_size: int = 1000 #size of validation set
    seed: int = 42 #random seed for method : summary pair shuffling into train and valid
    english_confidence_threshold: float = 0.70 #confidence threshold for english language checks on mined method summaries using lingua lanugage detector
    repo_metadata_confidence_threshold: float = 0.60 #confidence threshold for english language checks on repo level metadata (description, READEME)
    readme_max_chars: int = 4000 #max characters of readme used when checking language
    comment_miss_audit_size: int = 50 #number of entries in missing comment audit
    filter_drop_audit_size: int = 50 #number of entries in filter (dedup, invalid method, etc.) drop audit
    clone_dir: str = str(DEFAULT_DATA_DIR / 'java_summary_repos') #destination directory for cloned repos
    output_dir: str = str(DEFAULT_DATA_DIR / 'code_summarization_dataset') #desitination directory for our code summarization dataset files
    audit_dir : str = str(DEFAULT_DATA_DIR / 'dataset_construction_audits') #desitination directory for audits of various stages of dataset construction
    backfill_dataset_name : str = str('google/code_x_glue_ct_code_to_text')
    backfill_dataset_split : str = str('java')
    backfill_dataset_split : str = str('train')

config = DatasetConfig()
config


/Users/sambennett/Desktop/CSCI555/assignment_2/code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetConfig(num_repos=700, min_stars=100, max_search_pages=10, max_files_per_repo=40, train_size=50000, val_size=1000, seed=42, english_confidence_threshold=0.7, repo_metadata_confidence_threshold=0.6, readme_max_chars=4000, comment_miss_audit_size=50, filter_drop_audit_size=50, clone_dir='/Users/sambennett/Desktop/CSCI555/assignment_2/data/java_summary_repos', output_dir='/Users/sambennett/Desktop/CSCI555/assignment_2/data/code_summarization_dataset', audit_dir='/Users/sambennett/Desktop/CSCI555/assignment_2/data/dataset_construction_audits')

## Setup Repo Mining Environment

In [3]:
def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(path_value).expanduser()
    if path.is_absolute():
        return path
    return (PROJECT_ROOT / path).resolve()


def build_language_detector():
    return LanguageDetectorBuilder.from_languages(
        Language.ENGLISH,
        Language.GERMAN,
        Language.FRENCH,
        Language.SPANISH,
        Language.PORTUGUESE,
        Language.ITALIAN,
    ).build()

randomizer = random.Random(config.seed)
clone_dir = resolve_project_path(config.clone_dir)
output_dir = resolve_project_path(config.output_dir)
audit_dir = resolve_project_path(config.audit_dir)
language_detector = build_language_detector()
clone_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)

## Fetch Git Repos That Have English Metadata (Descriptions, README)

In [ ]:
def build_github_headers() -> dict[str, str]:
    headers = {"Accept": "application/vnd.github+json"}

    #local notebook usage
    github_token = os.getenv("GITHUB_TOKEN")

    if github_token:
        headers["Authorization"] = f"Bearer {github_token}"
        #print("GitHub token found and will be used for API requests.")
    else:
        print("GitHub token NOT found. API requests will be unauthenticated and subject to lower rate limits.")
    return headers

def fetch_top_java_repos_page(page: int, min_stars: int, per_page: int = 100) -> list[dict]:
  response = requests.get(
      "https://api.github.com/search/repositories",
      params={
          "q": f"language:java stars:>{min_stars}",
          "sort": "stars",
          "order": "desc",
          "per_page": per_page,
          "page": page,
      },
      headers=build_github_headers(),
      timeout=30,
  )
  response.raise_for_status()

  repos = []
  for item in response.json().get("items", []):
      if item.get("fork", False):
          continue
      repos.append(
          {
              "full_name": item["full_name"],
              "clone_url": item["clone_url"],
              "stars": item["stargazers_count"],
              "description": item.get("description") or "",
          }
      )

  return repos

def fetch_repo_readme(owner_repo: str, max_chars: int) -> str | None:
    response = requests.get(
        f"https://api.github.com/repos/{owner_repo}/readme",
        headers=build_github_headers(),
        timeout=30,
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()
    payload = response.json()
    encoded_content = payload.get("content")
    encoding = payload.get("encoding")
    if not encoded_content or encoding != "base64":
        return None

    try:
        decoded = base64.b64decode(encoded_content).decode("utf-8", errors="replace")
    except Exception:
        return None

    return decoded[:max_chars]

def contains_non_ascii(text: str) -> bool:
    try:
        text.encode("ascii")
        return False
    except UnicodeEncodeError:
        return True

def classify_english_text(text: str, detector, confidence_threshold: float, min_words: int = 3) -> str:
  if not text:
      return "ambiguous"

  normalized = re.sub(r"\s+", " ", text).strip()
  if not normalized:
      return "ambiguous"

  if contains_non_ascii(normalized):
      try:
          detected = detector.detect_language_of(normalized)
          if detected == Language.ENGLISH:
              confidence_values = detector.compute_language_confidence_values(normalized)
              english_confidence = next(
                  (value.value for value in confidence_values if value.language == Language.ENGLISH),
                  0.0,
              )
              return "english" if english_confidence >= confidence_threshold else "ambiguous"
          return "non_english"
      except Exception:
          return "ambiguous"

  words = re.findall(r"[a-z']+", normalized.lower())
  if len(words) < min_words:
      return "ambiguous"

  try:
      detected = detector.detect_language_of(normalized)
      if detected != Language.ENGLISH:
          return "non_english"

      confidence_values = detector.compute_language_confidence_values(normalized)
      english_confidence = next(
          (value.value for value in confidence_values if value.language == Language.ENGLISH),
          0.0,
      )
      return "english" if english_confidence >= confidence_threshold else "ambiguous"
  except Exception:
      return "ambiguous"

def repo_metadata_is_english(repo: dict, detector, confidence_threshold: float, readme_max_chars: int) -> tuple[bool, str, bool, bool]:
    description_checked = True
    readme_checked = False

    description_status = classify_english_text(
        repo.get("description", ""),
        detector=detector,
        confidence_threshold=confidence_threshold,
        min_words=3,
    )
    if description_status == "english":
        return True, "english_description", description_checked, readme_checked
    if description_status == "non_english":
        return False, "non_english_description", description_checked, readme_checked

    readme_checked = True
    try:
        readme_text = fetch_repo_readme(repo["full_name"], max_chars=readme_max_chars)
    except requests.RequestException:
        return True, "ambiguous_readme_fetch_failed", description_checked, readme_checked

    if not readme_text:
        return True, "ambiguous_no_readme", description_checked, readme_checked

    readme_status = classify_english_text(
        readme_text,
        detector=detector,
        confidence_threshold=confidence_threshold,
        min_words=10,
    )
    if readme_status == "english":
        return True, "english_readme", description_checked, readme_checked
    if readme_status == "non_english":
        return False, "non_english_readme", description_checked, readme_checked
    return True, "ambiguous_readme", description_checked, readme_checked

def collect_screened_repos(
    num_repos: int,
    min_stars: int,
    detector,
    metadata_confidence_threshold: float,
    max_search_pages: int,
    readme_max_chars: int,
) -> tuple[list[dict], dict]:
  screened_repos = []
  stats = {
      "candidate_repos_examined": 0,
      "repos_passed_metadata_screen": 0,
      "repos_skipped_non_english_description": 0,
      "repos_checked_via_readme": 0,
      "repos_skipped_non_english_readme": 0,
      "pages_fetched": 0,
  }

  for page in range(1, max_search_pages + 1):
      page_repos = fetch_top_java_repos_page(page=page, min_stars=min_stars)
      stats["pages_fetched"] += 1
      if not page_repos:
          break

      for repo in page_repos:
          if len(screened_repos) >= num_repos:
              break

          stats["candidate_repos_examined"] += 1
          keep_repo, metadata_status, description_checked, readme_checked = repo_metadata_is_english(
              repo=repo,
              detector=detector,
              confidence_threshold=metadata_confidence_threshold,
              readme_max_chars=readme_max_chars,
          )
          repo["metadata_english_status"] = metadata_status
          repo["description_checked"] = description_checked
          repo["readme_checked"] = readme_checked
          repo["metadata_skip_reason"] = "" if keep_repo else metadata_status

          if readme_checked:
              stats["repos_checked_via_readme"] += 1
          if metadata_status == "non_english_description":
              stats["repos_skipped_non_english_description"] += 1
          if metadata_status == "non_english_readme":
              stats["repos_skipped_non_english_readme"] += 1

          if keep_repo:
              screened_repos.append(repo)
              stats["repos_passed_metadata_screen"] += 1

      if len(screened_repos) >= num_repos:
          break

  return screened_repos, stats


print("Fetching and screening repository metadata from GitHub...")
repos, repo_collection_stats = collect_screened_repos(
    num_repos=config.num_repos,
    min_stars=config.min_stars,
    detector=language_detector,
    metadata_confidence_threshold=config.repo_metadata_confidence_threshold,
    max_search_pages=config.max_search_pages,
    readme_max_chars=config.readme_max_chars,
)
print(f"Collected {len(repos)} repositories that passed metadata English screening.")
print(
    "Using lingua for English detection "
    f"(threshold={config.english_confidence_threshold:.2f})."
)
print(
    "Using repo metadata screening threshold "
    f"{config.repo_metadata_confidence_threshold:.2f}."
)
if len(repos) < config.num_repos:
    print(
        f"Only {len(repos)} repos passed metadata screening before search exhaustion "
        f"or hitting the max page limit ({config.max_search_pages})."
    )

Fetching and screening repository metadata from GitHub...
Collected 700 repositories that passed metadata English screening.
Using lingua for English detection (threshold=0.70).
Using repo metadata screening threshold 0.60.


## Clone Selected Repositories

In [ ]:
def clone_repo(clone_url: str, dest_dir: str) -> bool:
  try:
      if os.path.exists(dest_dir):
          shutil.rmtree(dest_dir)

      cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
      result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
      return result.returncode == 0
  except subprocess.TimeoutExpired:
      print(f"  Timeout cloning {clone_url}")
      return False
  except Exception as exc:
      print(f"  Error cloning {clone_url}: {exc}")
      return False


cloned_repos = []
failed_repos = []
repo_stats = {}

print("Cloning repositories...")
for index, repo in enumerate(repos, start=1):
    repo_name = repo["full_name"]
    safe_name = repo_name.replace("/", "_")
    destination = clone_dir / safe_name

    if destination.is_dir() and any(destination.iterdir()):
        print(f"[{index}/{len(repos)}] Reusing {repo_name}")
        cloned_repos.append({"repo_name": repo_name, "local_path": str(destination)})
        repo_stats[repo_name] = {
            "repo": repo_name,
            "stars": repo["stars"],
            "description_checked": repo.get("description_checked", False),
            "readme_checked": repo.get("readme_checked", False),
            "metadata_english_status": repo.get("metadata_english_status", "unknown"),
            "metadata_skip_reason": repo.get("metadata_skip_reason", ""),
            "selected_files": 0,
            "total_methods_seen": 0,
            "methods_with_comments": 0,
            "incomplete_dropped": 0,
            "english_dropped": 0,
            "duplicate_dropped": 0,
            "unique_pairs_kept": 0,
        }
        continue

    print(f"[{index}/{len(repos)}] Cloning {repo_name}")
    if clone_repo(repo["clone_url"], str(destination)):
        cloned_repos.append({"repo_name": repo_name, "local_path": str(destination)})
        repo_stats[repo_name] = {
            "repo": repo_name,
            "stars": repo["stars"],
            "description_checked": repo.get("description_checked", False),
            "readme_checked": repo.get("readme_checked", False),
            "metadata_english_status": repo.get("metadata_english_status", "unknown"),
            "metadata_skip_reason": repo.get("metadata_skip_reason", ""),
            "selected_files": 0,
            "total_methods_seen": 0,
            "methods_with_comments": 0,
            "incomplete_dropped": 0,
            "english_dropped": 0,
            "duplicate_dropped": 0,
            "unique_pairs_kept": 0,
        }
    else:
        failed_repos.append(repo_name)

Cloning repositories...
[1/700] Reusing krahets/hello-algo
[2/700] Reusing iluwatar/java-design-patterns
[3/700] Reusing spring-projects/spring-boot
[4/700] Reusing doocs/advanced-java
[5/700] Reusing MisterBooo/LeetCodeAnimation
[6/700] Reusing elastic/elasticsearch
[7/700] Reusing NationalSecurityAgency/ghidra
[8/700] Reusing TheAlgorithms/Java
[9/700] Reusing kdn251/interviews
[10/700] Reusing spring-projects/spring-framework
[11/700] Reusing termux/termux-app
[12/700] Reusing google/guava
[13/700] Reusing dbeaver/dbeaver
[14/700] Reusing ReactiveX/RxJava
[15/700] Reusing skylot/jadx
[16/700] Reusing apache/dubbo
[17/700] Reusing PhilJay/MPAndroidChart
[18/700] Reusing TeamNewPipe/NewPipe
[19/700] Reusing eugenp/tutorials
[20/700] Reusing alibaba/arthas
[21/700] Reusing doocs/leetcode
[22/700] Reusing airbnb/lottie-android
[23/700] Reusing ashishps1/awesome-system-design-resources
[24/700] Reusing bumptech/glide
[25/700] Reusing netty/netty
[26/700] Reusing SeleniumHQ/selenium
[27/7

## Select Java Files

In [ ]:
def find_java_files(repo_path: str) -> list[str]:
  java_files = []
  exclude_patterns = ["test", "tests", "example", "examples", "sample", "demo", "generated"]

  for root, _, files in os.walk(repo_path):
      root_lower = root.lower()
      if any(pattern in root_lower for pattern in exclude_patterns):
          continue

      for file_name in files:
          if file_name.endswith(".java"):
              java_files.append(os.path.join(root, file_name))

  return java_files


def select_java_files(java_files: list[str], max_files: int) -> list[str]:
  random.seed(42)
  if len(java_files) <= max_files:
      return java_files
  return random.sample(java_files, max_files)

selected_files = []
print(f"Selecting up to {config.max_files_per_repo} Java files per repository...")
for repo in cloned_repos:
  java_files = find_java_files(repo["local_path"])
  chosen_files = select_java_files(java_files, config.max_files_per_repo)
  repo_stats[repo["repo_name"]]["selected_files"] = len(chosen_files)
  selected_files.extend((repo["repo_name"], file_path) for file_path in chosen_files)

print(f"Selected {len(selected_files)} Java files across {len(cloned_repos)} repositories.")

Selecting up to 40 Java files per repository...
Selected 22969 Java files across 700 repositories.


## Extract Java Methods:
Here we extract java methods from the selected java methods that contain docstring or otherwise top-level comments.



In [ ]:
def read_file_content(file_path: str) -> str | None:
  encodings = ["utf-8", "latin-1", "cp1252"]
  for encoding in encodings:
      try:
          with open(file_path, "r", encoding=encoding) as handle:
              return handle.read()
      except UnicodeDecodeError:
          continue
  return None


def extract_method_source(source_code: str, method_node, lines: list[str]) -> str | None:
  try:
      start_line = method_node.position.line - 1
      brace_count = 0
      started = False
      end_line = start_line

      for index in range(start_line, len(lines)):
          for char in lines[index]:
              if char == "{":
                  brace_count += 1
                  started = True
              elif char == "}":
                  brace_count -= 1

          if started and brace_count == 0:
              end_line = index
              break

      return "\n".join(lines[start_line:end_line + 1])
  except Exception:
      return None


def clean_comment_text(raw_comment: str) -> str:
  cleaned_lines = []
  for line in raw_comment.splitlines():
      line = re.sub(r"^\s*/\*\*?", "", line)
      line = re.sub(r"\*/\s*$", "", line)
      line = re.sub(r"^\s*\*\s?", "", line)
      line = re.sub(r"^\s*//\s?", "", line)
      line = line.strip()

      if not line:
          if cleaned_lines:
              break
          continue
      if line.startswith("@"):
          break
      cleaned_lines.append(line)

  return re.sub(r"\s+", " ", " ".join(cleaned_lines)).strip()


def extract_top_level_comment(lines: list[str], method_start_line: int) -> str | None:
  index = method_start_line - 2

  while index >= 0:
      stripped = lines[index].strip()
      if not stripped or stripped.startswith("@"):
          index -= 1
          continue
      break

  if index < 0:
      return None

  stripped = lines[index].strip()
  if stripped.endswith("*/") and not stripped.startswith("//"):
      comment_lines = []
      while index >= 0:
          comment_lines.append(lines[index])
          if "/*" in lines[index]:
              break
          index -= 1
      if index < 0:
          return None
      comment_lines.reverse()
      return clean_comment_text("\n".join(comment_lines))

  if stripped.startswith("//"):
      comment_lines = []
      while index >= 0 and lines[index].strip().startswith("//"):
          comment_lines.append(lines[index])
          index -= 1
      comment_lines.reverse()
      return clean_comment_text("\n".join(comment_lines))

  return None


def extract_leading_body_comment(method_source: str) -> str | None:
  lines = method_source.splitlines()
  body_started = False
  index = 0

  while index < len(lines):
      current_line = lines[index]

      if not body_started:
          if "{" not in current_line:
              index += 1
              continue
          body_started = True
          current_line = current_line.split("{", 1)[1]

      stripped = current_line.strip()
      if not stripped:
          index += 1
          continue

      if stripped.startswith("//"):
          comment_lines = []
          while index < len(lines) and lines[index].strip().startswith("//"):
              comment_lines.append(lines[index])
              index += 1
          return clean_comment_text("\n".join(comment_lines))

      if stripped.startswith("/*"):
          comment_lines = [current_line]
          if "*/" in stripped:
              return clean_comment_text("\n".join(comment_lines))

          index += 1
          while index < len(lines):
              comment_lines.append(lines[index])
              if "*/" in lines[index]:
                  break
              index += 1
          return clean_comment_text("\n".join(comment_lines))

      return None

  return None


def extract_method_comment(lines: list[str], method_start_line: int, method_source: str) -> str | None:
  top_level_comment = extract_top_level_comment(lines, method_start_line)
  if top_level_comment:
      return top_level_comment
  return extract_leading_body_comment(method_source)

def build_comment_miss_audit_entry(repo_name: str, file_path: str, method_name: str, method_start_line: int, lines: list[str]) -> dict:
    start_index = max(0, method_start_line - 6)
    context_lines = []
    for line_number in range(start_index, method_start_line):
        context_lines.append(f"{line_number + 1}: {lines[line_number]}")

    signature_line = ""
    if 0 <= method_start_line - 1 < len(lines):
        signature_line = lines[method_start_line - 1].strip()

    return {
        "repo": repo_name,
        "file": file_path,
        "method_name": method_name,
        "method_start_line": method_start_line,
        "signature_line": signature_line,
        "preceding_context": context_lines,
    }


def add_to_reservoir_sample(reservoir: list[dict], candidate: dict, max_size: int, items_seen: int, randomizer: random.Random) -> int:
    items_seen += 1
    if max_size <= 0:
        return items_seen
    if len(reservoir) < max_size:
        reservoir.append(candidate)
        return items_seen

    replacement_index = randomizer.randrange(items_seen)
    if replacement_index < max_size:
        reservoir[replacement_index] = candidate
    return items_seen


def extract_methods_with_comments(
    file_path: str,
    repo_name: str,
    comment_miss_audit: list[dict] | None = None,
    comment_miss_audit_size: int = 0,
    comment_miss_count: int = 0,
    audit_randomizer: random.Random | None = None,
) -> tuple[list[dict], int, int]:
    extracted = []
    source_code = read_file_content(file_path)
    if source_code is None:
        return extracted, 0, comment_miss_count

    lines = source_code.splitlines()
    try:
        tree = javalang.parse.parse(source_code)
    except (
        javalang.parser.JavaSyntaxError,
        javalang.tokenizer.LexerError,
        TypeError,
        IndexError,
        ValueError,
    ):
        return extracted, 0, comment_miss_count

    total_methods_in_file = 0
    try:
        for _, node in tree.filter(javalang.tree.MethodDeclaration):
            total_methods_in_file += 1
            if not node.position:
                continue

            method_source = extract_method_source(source_code, node, lines)
            if not method_source:
                continue

            comment = extract_method_comment(lines, node.position.line, method_source)
            if not comment:
                if comment_miss_audit is not None and audit_randomizer is not None:
                    comment_miss_count = add_to_reservoir_sample(
                        reservoir=comment_miss_audit,
                        candidate=build_comment_miss_audit_entry(
                            repo_name=repo_name,
                            file_path=file_path,
                            method_name=node.name,
                            method_start_line=node.position.line,
                            lines=lines,
                        ),
                        max_size=config.comment_miss_audit_size,
                        items_seen=comment_miss_count,
                        randomizer=audit_randomizer,
                    )
                continue

            extracted.append(
                {
                    "repo": repo_name,
                    "file": os.path.basename(file_path),
                    "method_name": node.name,
                    "source": method_source,
                    "summary": comment,
                }
            )
    except RecursionError:
        print(f"Skipping {file_path}: javalang AST traversal exceeded recursion depth.")
        return [], 0, comment_miss_count
    except Exception as exc:
        print(f"Skipping {file_path}: unexpected AST traversal error: {exc}")
        return [], 0, comment_miss_count

    return extracted, total_methods_in_file, comment_miss_count


extracted_methods = []
total_methods_seen = 0
comment_miss_audit = []
total_comment_misses_seen = 0
for index, (repo_name, file_path) in enumerate(selected_files, start=1):
    if index % 100 == 0 or index == len(selected_files):
        print(f"Processed {index}/{len(selected_files)} files for method extraction.")
    extracted_from_file, method_count, total_comment_misses_seen = extract_methods_with_comments(
        file_path,
        repo_name,
        comment_miss_audit=comment_miss_audit,
        comment_miss_audit_size=config.comment_miss_audit_size,
        comment_miss_count=total_comment_misses_seen,
        audit_randomizer=randomizer,
    )
    total_methods_seen += method_count
    repo_stats[repo_name]["total_methods_seen"] += method_count
    repo_stats[repo_name]["methods_with_comments"] += len(extracted_from_file)
    extracted_methods.extend(extracted_from_file)

Processed 100/22969 files for method extraction.
Processed 200/22969 files for method extraction.
Processed 300/22969 files for method extraction.
Processed 400/22969 files for method extraction.
Processed 500/22969 files for method extraction.
Processed 600/22969 files for method extraction.
Processed 700/22969 files for method extraction.
Processed 800/22969 files for method extraction.
Processed 900/22969 files for method extraction.
Processed 1000/22969 files for method extraction.
Processed 1100/22969 files for method extraction.
Processed 1200/22969 files for method extraction.
Processed 1300/22969 files for method extraction.
Processed 1400/22969 files for method extraction.
Processed 1500/22969 files for method extraction.
Processed 1600/22969 files for method extraction.
Processed 1700/22969 files for method extraction.
Processed 1800/22969 files for method extraction.
Processed 1900/22969 files for method extraction.
Processed 2000/22969 files for method extraction.
Processed

## Normalize and Filter Extracted Methods/Summaries

First, we guarantee our extracted methods and summaries are flattend into single lines and that our summaries are completely lowercase. Second, we filter out methods that are not clean/complete and summaries that are not english. Finally, we filter out all duplicate methods.

In [ ]:
def normalize_method_source(source: str) -> str:
    return re.sub(r"\s+", " ", source).strip()

def passes_method_cleanliness_check(source: str) -> bool:
    if not source or "{" not in source or not source.endswith("}"):
        return False

    brace_balance = 0
    for char in source:
        if char == "{":
            brace_balance += 1
        elif char == "}":
            brace_balance -= 1
            if brace_balance < 0:
                return False

    return brace_balance == 0

def looks_like_code_noise(text: str) -> bool:
    if re.search(r"[{};]|::|->", text):
        return True

    tokens = text.split()
    if not tokens:
        return True

    noisy_tokens = sum(
        1
        for token in tokens
        if "_" in token
        or "/" in token
        or "\\" in token
        or ("." in token and not token.rstrip(".").isalpha())
        or re.search(r"[a-z][A-Z]", token) is not None
    )
    return noisy_tokens > max(2, len(tokens) // 2)

#these summary verbs are used in an simple intial language check of summaries
SHORT_SUMMARY_VERBS = {
    "add", "build", "calculate", "check", "close", "convert", "create", "delete",
    "direct", "fetch", "find", "get", "handle", "initialize", "load", "make",
    "merge", "open", "parse", "print", "read", "remove", "return", "save", "set",
    "sort", "translate", "update", "validate", "write",
}

def is_short_english_summary(text: str) -> bool:
    lowered = text.lower().strip()
    if not lowered or looks_like_code_noise(lowered):
        return False

    words = re.findall(r"[a-z']+", lowered)
    if not 2 <= len(words) <= 5:
        return False
    if any(len(word) == 1 and word not in {"a", "i"} for word in words):
        return False
    if words[0] not in SHORT_SUMMARY_VERBS:
        return False

    alpha_chars = sum(1 for char in lowered if char.isalpha())
    visible_chars = sum(1 for char in lowered if not char.isspace())
    if visible_chars == 0 or alpha_chars / visible_chars < 0.65:
        return False
    return True


def passes_basic_english_prefilter(text: str) -> bool:
    if not text or contains_non_ascii(text):
        return False
    if looks_like_code_noise(text):
        return False

    words = re.findall(r"[a-z']+", text.lower())
    if len(words) < 2:
        return False

    alpha_chars = sum(1 for char in text if char.isalpha())
    visible_chars = sum(1 for char in text if not char.isspace())
    if visible_chars == 0 or alpha_chars / visible_chars < 0.5:
        return False

    suspicious_tokens = sum(
        1 for token in text.split() if any(ch.isdigit() for ch in token) or "_" in token or "/" in token
    )
    return suspicious_tokens <= max(2, len(words) // 2)


def is_probably_english(text: str, detector, confidence_threshold: float = 0.80) -> bool:
    if not passes_basic_english_prefilter(text):
        return False
    if is_short_english_summary(text):
        return True

    try:
        detected_language = detector.detect_language_of(text)
        if detected_language != Language.ENGLISH:
            return False
        confidence_values = detector.compute_language_confidence_values(text)
        english_confidence = next(
            (value.value for value in confidence_values if value.language == Language.ENGLISH),
            0.0,
        )
        return english_confidence >= confidence_threshold
    except Exception:
        return True

methods_with_comments = len(extracted_methods)
unique_method_pairs: dict[str, str] = {}
deduped_examples = []
incomplete_dropped = 0
english_dropped = 0
duplicate_dropped = 0
incomplete_drop_audit = []
english_drop_audit = []

print("normalizing and filtering method code/summary")
index = 0
for item in extracted_methods:
    index += 1
    if index % 1000 == 0 or index == len(extracted_methods):
        print(f"Processed {index}/{len(extracted_methods)} method/summary pairs for normalization and filtering.")

    normalized_code = normalize_method_source(item["source"])
    normalized_summary = item["summary"].lower()

    if not passes_method_cleanliness_check(normalized_code):
        incomplete_dropped += 1
        repo_stats[item["repo"]]["incomplete_dropped"] += 1
        if len(incomplete_drop_audit) < config.filter_drop_audit_size:
            incomplete_drop_audit.append(
                {
                    "repo": item["repo"],
                    "file": item["file"],
                    "method_name": item["method_name"],
                    "summary": normalized_summary,
                    "code": normalized_code,
                    "reason": "failed brace-based completeness check",
                }
            )
        continue

    if not is_probably_english(
        normalized_summary,
        detector=language_detector,
        confidence_threshold=config.english_confidence_threshold,
    ):
        english_dropped += 1
        repo_stats[item["repo"]]["english_dropped"] += 1
        if len(english_drop_audit) < config.filter_drop_audit_size:
            english_drop_audit.append(
                {
                    "repo": item["repo"],
                    "file": item["file"],
                    "method_name": item["method_name"],
                    "summary": normalized_summary,
                    "code": normalized_code,
                    "reason": "failed lingua or summary-quality filter",
                }
            )
        continue

    if normalized_code in unique_method_pairs:
        duplicate_dropped += 1
        repo_stats[item["repo"]]["duplicate_dropped"] += 1
        continue

    unique_method_pairs[normalized_code] = normalized_summary
    deduped_examples.append({"code": normalized_code, "summary": normalized_summary})
    repo_stats[item["repo"]]["unique_pairs_kept"] += 1

normalizing and filtering method code/summary
Processed 1000/55904 method/summary pairs for normalization and filtering.
Processed 2000/55904 method/summary pairs for normalization and filtering.
Processed 3000/55904 method/summary pairs for normalization and filtering.
Processed 4000/55904 method/summary pairs for normalization and filtering.
Processed 5000/55904 method/summary pairs for normalization and filtering.
Processed 6000/55904 method/summary pairs for normalization and filtering.
Processed 7000/55904 method/summary pairs for normalization and filtering.
Processed 8000/55904 method/summary pairs for normalization and filtering.
Processed 9000/55904 method/summary pairs for normalization and filtering.
Processed 10000/55904 method/summary pairs for normalization and filtering.
Processed 11000/55904 method/summary pairs for normalization and filtering.
Processed 12000/55904 method/summary pairs for normalization and filtering.
Processed 13000/55904 method/summary pairs for norm

## Summary of Repo Mining

In [ ]:
print("\nSummary report")
print(f"  Candidate repos examined:             {repo_collection_stats['candidate_repos_examined']}")
print(f"  Repos passed metadata screen:         {repo_collection_stats['repos_passed_metadata_screen']}")
print(f"  Repos skipped by description:         {repo_collection_stats['repos_skipped_non_english_description']}")
print(f"  Repos checked via README:             {repo_collection_stats['repos_checked_via_readme']}")
print(f"  Repos skipped by README:              {repo_collection_stats['repos_skipped_non_english_readme']}")
print(f"  Search pages fetched:                 {repo_collection_stats['pages_fetched']}")
print(f"  Repositories requested:               {config.num_repos}")
print(f"  Repositories fetched:                 {len(repos)}")
print(f"  Repositories cloned or reused:        {len(cloned_repos)}")
print(f"  Repository clone failures:            {len(failed_repos)}")
print(f"  Java files selected:                  {len(selected_files)}")
print(f"  Total methods seen in selected files: {total_methods_seen}")
print(f"  Methods extracted with comments:      {methods_with_comments}")
print(f"  Incomplete methods dropped:           {incomplete_dropped}")
print(f"  Methods dropped by English filter:    {english_dropped}")
print(f"  Duplicate methods dropped:            {duplicate_dropped}")
print(f"  Unique method-summary pairs kept:     {len(deduped_examples)}")


Summary report
  Candidate repos examined:             879
  Repos passed metadata screen:         700
  Repos skipped by description:         177
  Repos checked via README:             134
  Repos skipped by README:              2
  Search pages fetched:                 9
  Repositories requested:               700
  Repositories fetched:                 700
  Repositories cloned or reused:        700
  Repository clone failures:            0
  Java files selected:                  22969
  Total methods seen in selected files: 193139
  Methods extracted with comments:      55904
  Incomplete methods dropped:           7460
  Methods dropped by English filter:    24227
  Duplicate methods dropped:            983
  Unique method-summary pairs kept:     23234


## Create Dropped Sample Audit Files

In [ ]:
def save_comment_miss_audit(audit_entries: list[dict], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as audit_file:
        for index, entry in enumerate(audit_entries, start=1):
            audit_file.write(f"Example {index}\n")
            audit_file.write(f"Repo: {entry['repo']}\n")
            audit_file.write(f"File: {entry['file']}\n")
            audit_file.write(f"Method: {entry['method_name']}\n")
            audit_file.write(f"Method start line: {entry['method_start_line']}\n")
            audit_file.write(f"Signature: {entry['signature_line']}\n")
            audit_file.write("Preceding context:\n")
            for line in entry["preceding_context"]:
                audit_file.write(f"{line}\n")
            audit_file.write("\n")


def save_drop_audit(drop_entries: list[dict], output_path: Path, title: str) -> None:
    with output_path.open("w", encoding="utf-8") as audit_file:
        for index, entry in enumerate(drop_entries, start=1):
            audit_file.write(f"{title} Example {index}\n")
            audit_file.write(f"Repo: {entry['repo']}\n")
            audit_file.write(f"File: {entry['file']}\n")
            audit_file.write(f"Method: {entry['method_name']}\n")
            audit_file.write(f"Summary: {entry['summary']}\n")
            if "reason" in entry:
                audit_file.write(f"Reason: {entry['reason']}\n")
            audit_file.write(f"Code: {entry['code']}\n\n")


def save_repo_yield_report(repo_rows: list[dict], output_path: Path) -> None:
    fieldnames = [
        "repo",
        "stars",
        "description_checked",
        "readme_checked",
        "metadata_english_status",
        "metadata_skip_reason",
        "selected_files",
        "total_methods_seen",
        "methods_with_comments",
        "incomplete_dropped",
        "english_dropped",
        "duplicate_dropped",
        "unique_pairs_kept",
    ]
    with output_path.open("w", encoding="utf-8", newline="") as report_file:
        writer = csv.DictWriter(report_file, fieldnames=fieldnames)
        writer.writeheader()
        for row in sorted(repo_rows, key=lambda item: item["unique_pairs_kept"], reverse=True):
            writer.writerow(row)

def save_examples_jsonl(examples: list[dict], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as output_file:
        for example in examples:
            output_file.write(json.dumps(example, ensure_ascii=False) + "\n")


comment_miss_audit_path = audit_dir / "comment_miss_audit.txt"
save_comment_miss_audit(comment_miss_audit, comment_miss_audit_path)
incomplete_audit_path = audit_dir / "incomplete_method_audit.txt"
english_audit_path = audit_dir / "english_filter_audit.txt"
repo_yield_report_path = audit_dir / "repo_yield_report.csv"
mined_examples_path = audit_dir / "mined_method_summary_pairs.jsonl"
save_examples_jsonl(deduped_examples, mined_examples_path)
save_drop_audit(incomplete_drop_audit, incomplete_audit_path, "Incomplete")
save_drop_audit(english_drop_audit, english_audit_path, "English Filter")
save_repo_yield_report(list(repo_stats.values()), repo_yield_report_path)
print(f"  Comment miss audit examples saved:    {len(comment_miss_audit)}")
print(f"  Total comment misses observed:        {total_comment_misses_seen}")
print(f"  Comment miss audit file:              {comment_miss_audit_path}")
print(f"  Incomplete drop audit file:           {incomplete_audit_path}")
print(f"  English filter audit file:            {english_audit_path}")
print(f"  Repo yield report:                    {repo_yield_report_path}")
print(f"  Mined pair snapshot:                  {mined_examples_path}")

  Comment miss audit examples saved:    50
  Total comment misses observed:        137235
  Comment miss audit file:              /Users/sambennett/desktop/CSCI555/assignment_2/data/code_summarization_dataset/comment_miss_audit.txt
  Incomplete drop audit file:           /Users/sambennett/desktop/CSCI555/assignment_2/data/code_summarization_dataset/incomplete_method_audit.txt
  English filter audit file:            /Users/sambennett/desktop/CSCI555/assignment_2/data/code_summarization_dataset/english_filter_audit.txt
  Repo yield report:                    /Users/sambennett/desktop/CSCI555/assignment_2/data/code_summarization_dataset/repo_yield_report.csv


## Insufficient Sample Collection Handling:
If we have failed to reach desired 50k train, 1k validation goal, backfill method/summary pairs from CODEXGulue code_to_text dataset.

In [ ]:
from datasets import load_dataset

def get_codexglue_summary(record: dict) -> str:
    # Prefer the raw docstring field used by the CodeXGLUE code-to-text dataset.
    summary = record.get("docstring") or record.get("summary") or ""
    if summary:
        return summary

    docstring_tokens = record.get("docstring_tokens")
    if isinstance(docstring_tokens, list):
        return " ".join(str(token) for token in docstring_tokens)
    return ""


def backfill_from_codexglue(
    deduped_examples: list[dict],
    unique_method_pairs: dict[str, str],
    target_total: int,
    detector,
    confidence_threshold: float,
    dataset_name: str,
    dataset_config: str,
    dataset_split: str,
) -> tuple[list[dict], int, int, int]:
    # Fill any shortfall with normalized CodeXGLUE method-summary pairs, deduped by method source.
    examples_added = 0
    duplicates_skipped = 0
    filtered_skipped = 0

    if len(deduped_examples) >= target_total:
        return deduped_examples, examples_added, duplicates_skipped, filtered_skipped

    dataset = load_dataset(dataset_name, dataset_config, split=dataset_split)
    for record in dataset:
        if len(deduped_examples) >= target_total:
            break

        normalized_code = normalize_method_source(record.get("code", ""))
        normalized_summary = clean_comment_text(get_codexglue_summary(record)).lower()

        if not passes_method_cleanliness_check(normalized_code):
            filtered_skipped += 1
            continue

        if not is_probably_english(
            normalized_summary,
            detector=detector,
            confidence_threshold=confidence_threshold,
        ):
            filtered_skipped += 1
            continue

        if normalized_code in unique_method_pairs:
            duplicates_skipped += 1
            continue

        unique_method_pairs[normalized_code] = normalized_summary
        deduped_examples.append({"code": normalized_code, "summary": normalized_summary})
        examples_added += 1

    return deduped_examples, examples_added, duplicates_skipped, filtered_skipped


required_examples = config.train_size + config.val_size
codexglue_added = 0
codexglue_duplicates_skipped = 0
codexglue_filtered_skipped = 0
if len(deduped_examples) < required_examples:
    print(
        f"Backfilling {required_examples - len(deduped_examples)} missing pairs "
        "from CodeXGLUE code-to-text."
    )
    deduped_examples, codexglue_added, codexglue_duplicates_skipped, codexglue_filtered_skipped = backfill_from_codexglue(
        deduped_examples=deduped_examples,
        unique_method_pairs=unique_method_pairs,
        target_total=required_examples,
        detector=language_detector,
        confidence_threshold=config.english_confidence_threshold,
        dataset_name=config.backfill_dataset_name,
        dataset_config=config.backfill_dataset_config,
        dataset_split=config.backfill_dataset_split,
    )
    print(f"  CodeXGLUE pairs added:                {codexglue_added}")
    print(f"  CodeXGLUE duplicates skipped:         {codexglue_duplicates_skipped}")
    print(f"  CodeXGLUE filtered pairs skipped:     {codexglue_filtered_skipped}")

final_examples_path = output_dir / "final_method_summary_pairs.jsonl"
save_examples_jsonl(deduped_examples, final_examples_path)
print(f"  Final pair snapshot:                  {final_examples_path}")

if len(deduped_examples) < required_examples:
    raise RuntimeError(
        f"Need at least {required_examples} examples for the requested split, "
        f"but only collected {len(deduped_examples)} after CodeXGLUE backfill."
    )

## Shuffle and Split Filtered Extracted Methods and Summaries into Train and Validation Sets

In [ ]:
def save_parallel_text(code_examples: list[dict], code_path: Path, summary_path: Path) -> None:
  with code_path.open("w", encoding="utf-8") as code_file:
      for example in code_examples:
          code_file.write(example["code"] + "\n")

  with summary_path.open("w", encoding="utf-8") as summary_file:
      for example in code_examples:
          summary_file.write(example["summary"] + "\n")


randomizer.shuffle(deduped_examples)
train_examples = deduped_examples[: config.train_size]
val_examples = deduped_examples[config.train_size : config.train_size + config.val_size]

save_parallel_text(train_examples, output_dir / "train_code.txt", output_dir / "train_summary.txt")
save_parallel_text(val_examples, output_dir / "val_code.txt", output_dir / "val_summary.txt")

print("\nSaved dataset files")
print(f"  Train code:       {output_dir / 'train_code.txt'}")
print(f"  Train summary:    {output_dir / 'train_summary.txt'}")
print(f"  Val code:         {output_dir / 'val_code.txt'}")
print(f"  Val summary:      {output_dir / 'val_summary.txt'}")
print(f"  Training pairs:   {len(train_examples)}")
print(f"  Validation pairs: {len(val_examples)}")

## Embed Train and Validation

We use the pretained embedding model codet5 to generate a vocab for our training and validation set, tokenize each sample pair and embed each token to provide our LSTM with baked in semantic meaning and relationships of tokens. This is all done with the given get_codet5_embeddings.py file. Note that summaries and code are embedded sepparately by codet5. 

In [11]:
embeddings_dir = output_dir / "embeddings"
embeddings_dir.mkdir(exist_ok=True)

train_code_in = str(output_dir / "train_code.txt")
train_code_out = str(embeddings_dir / "train_code.pt")

train_summary_in = str(output_dir / "train_summary.txt")
train_summary_out = str(embeddings_dir / "train_summary.pt")

val_code_in = str(output_dir / "val_code.txt")
val_code_out = str(embeddings_dir / "val_code.pt")

val_summary_in = str(output_dir / "val_summary.txt")
val_summary_out = str(embeddings_dir / "val_summary.pt")

In [8]:
!python get_codet5_embeddings.py --input "$train_code_in" --output "$train_code_out"
!python get_codet5_embeddings.py --input "$train_summary_in" --output "$train_summary_out" --max_length 128
!python get_codet5_embeddings.py --input "$val_code_in" --output "$val_code_out"
!python get_codet5_embeddings.py --input "$val_summary_in" --output "$val_summary_out" --max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 50000 samples from /Users/sambennett/Desktop/CSCI555/assignment_2/data/code_summarization_dataset/train_code.txt
Tokenizing: 100%|███████████████████████| 50000/50000 [00:10<00:00, 4926.50it/s]

Token length stats:
  Mean: 110.8
  Max:  512
  Min:  7

Saved to /Users/sambennett/Desktop/CSCI555/assignment_2/data/code_summarization_dataset/embeddings/train_code.pt
Loading tokenizer and model: Salesforce/codet5p-220m
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 50000 samples from /Users/sambennett/Desktop/CSCI555/assignment_2/data/code_summarization_dataset/train_summary.txt
Tokenizing: 100%|██████████████████████| 50000/50000 [00:02<00:00, 21587.49it/s]

Token length stats:
  Mean: 19.8
  Max:  128
  Min:  4

Saved to /Users/sambennett/Desktop/CSCI555/

# Train and Validate LSTM

## Import Dependencies and Setup Constants for Training and Validation

Note a smoke test built into the current code, if you would like to test training just for functionality and not for end results you can enable smoketest to true. 

In [47]:
import torch
import torch.nn as nn
import sacrebleu
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm
import numpy as np
import json
import os
from pathlib import Path

SMOKE_TEST = True #ensure this is set to false for full training
SMOKE_TRAIN_EXAMPLES = 16
SMOKE_VAL_EXAMPLES = 8
BATCH_SIZE = 4 if SMOKE_TEST else 32
MAX_LEN = 30 if SMOKE_TEST else 100
HIDDEN_DIM = 256
NUM_EPOCHS = 1 if SMOKE_TEST else 10
EVAL_STEPS = 2 if SMOKE_TEST else 500
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_code_path = embeddings_dir / "train_code.pt"
train_summary_path = embeddings_dir / "train_summary.pt"
val_code_path = embeddings_dir / "val_code.pt"
val_summary_path = embeddings_dir / "val_summary.pt"

## Initialize Dataloader and Load and Validate Datasets

In [14]:
class CodeSummaryDataset(Dataset):
    # Wrap aligned code/summary token-id sequences so the DataLoader can fetch
    # one training example at a time.
    def __init__(self, code_token_ids, summary_token_ids):
        if len(code_token_ids) != len(summary_token_ids):
            raise ValueError("Code and summary datasets must contain the same number of examples.")

        self.code_token_ids = code_token_ids
        self.summary_token_ids = summary_token_ids

    def __len__(self):
        return len(self.code_token_ids)

    def __getitem__(self, idx):
        # Convert one aligned pair of token-id lists into tensors. Padding is
        # handled later by the collate function, not here.
        src = torch.tensor(self.code_token_ids[idx], dtype=torch.long)
        tgt = torch.tensor(self.summary_token_ids[idx], dtype=torch.long)
        return src, tgt

def load_split_pt(code_path, summary_path):
    code_data = torch.load(code_path)
    summary_data = torch.load(summary_path)

    # The code and summary files must come from the same tokenizer/checkpoint so
    # the ids refer to the same vocabulary and embedding table.
    if code_data["tokenizer_name"] != summary_data["tokenizer_name"]:
        raise ValueError("Code and summary .pt files were built with different tokenizers.")
    if code_data["vocab_size"] != summary_data["vocab_size"]:
        raise ValueError("Code and summary .pt files have different vocabulary sizes.")
    if code_data["embedding_dim"] != summary_data["embedding_dim"]:
        raise ValueError("Code and summary .pt files have different embedding dimensions.")
    if code_data["pad_token_id"] != summary_data["pad_token_id"]:
        raise ValueError("Code and summary .pt files have different pad token ids.")
    if code_data["eos_token_id"] != summary_data["eos_token_id"]:
        raise ValueError("Code and summary .pt files have different eos token ids.")

    dataset = CodeSummaryDataset(code_data["token_ids"], summary_data["token_ids"])
    metadata = {
        "embedding_matrix": code_data["embedding_matrix"],
        "tokenizer_name": code_data["tokenizer_name"],
        "pad_id": code_data["pad_token_id"],
        "eos_id": code_data["eos_token_id"],
        "vocab_size": code_data["vocab_size"],
        "embedding_dim": code_data["embedding_dim"],
    }
    return dataset, metadata

def limit_dataset(dataset, max_examples):
    # Keep only a small deterministic prefix of examples for quick local smoke tests.
    if max_examples is None or max_examples >= len(dataset):
        return dataset
    
    return CodeSummaryDataset(
        dataset.code_token_ids[:max_examples],
        dataset.summary_token_ids[:max_examples],
    )

def make_collate_fn(pad_id):
    # DataLoader calls this on a list of examples. It pads all source sequences
    # to the same length and all target sequences to the same length so they can
    # be stacked into one batch tensor.
    def collate_fn(batch):
        src_batch, tgt_batch = zip(*batch)
        src_batch = pad_sequence(src_batch, batch_first=True, padding_value=pad_id)
        tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=pad_id)
        return src_batch, tgt_batch

    return collate_fn
    
# Load both splits and verify that the stored tokenizer metadata is consistent.
train_dataset, train_metadata = load_split_pt(train_code_path, train_summary_path)
val_dataset, val_metadata = load_split_pt(val_code_path, val_summary_path)

# Smoke-test mode shrinks the datasets and training schedule so the full
# pipeline can be checked quickly on a laptop.
if SMOKE_TEST:
    train_dataset = limit_dataset(train_dataset, SMOKE_TRAIN_EXAMPLES)
    val_dataset = limit_dataset(val_dataset, SMOKE_VAL_EXAMPLES)
    print(
        f"Running in SMOKE_TEST mode: "
        f"{len(train_dataset)} train examples, {len(val_dataset)} val examples, "
        f"batch_size={BATCH_SIZE}, epochs={NUM_EPOCHS}, eval_steps={EVAL_STEPS}, max_len={MAX_LEN}"
    )

if train_metadata["tokenizer_name"] != val_metadata["tokenizer_name"]:
    raise ValueError("Training and validation splits were built with different tokenizers.")
if train_metadata["pad_id"] != val_metadata["pad_id"]:
    raise ValueError("Training and validation splits have different pad ids.")
if train_metadata["eos_id"] != val_metadata["eos_id"]:
    raise ValueError("Training and validation splits have different eos ids.")

# Pull the special token ids from the saved preprocessing metadata.
pad_id = train_metadata["pad_id"]
eos_id = train_metadata["eos_id"]

# CodeT5 often uses the same special token at the front of decoder inputs.
# Until we add explicit tokenizer-driven decoding, reusing EOS as the start
# token keeps generation/training consistent with the stored token ids.
sos_id = eos_id

# Pad variable-length code and summary sequences into rectangular batches.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=make_collate_fn(pad_id),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_collate_fn(pad_id),
)

Running in SMOKE_TEST mode: 16 train examples, 8 val examples, batch_size=4, epochs=1, eval_steps=2, max_len=30


## Load matching tokenizer for validation decoding and SacreBlEU scoring

In [15]:
# Load the matching tokenizer once for validation decoding and SacreBLEU scoring.
tokenizer = AutoTokenizer.from_pretrained(
    train_metadata["tokenizer_name"],
    local_files_only=True,
)
bleu_metric = BLEU(max_ngram_order=1)

## Model Construction

In [17]:
# LSTM encoder-decoder model for code summarization.
# Source code tokens are embedded, encoded into a final hidden state, and that
# hidden state initializes the decoder, which generates summary tokens.
class Model(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        pad_id,
        eos_id,
        sos_id,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        dropout=0.2,
        freeze_embeddings=False,
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        # Persist special token ids on the model so training/inference do not
        # depend on hardcoded assumptions about the tokenizer.
        self.pad_id = pad_id #id of padding token used to guarantee every embded method code in batch has the same length
        self.eos_id = eos_id #id of end of sentence token appended each embeded method code
        self.sos_id = sos_id #id of start of sentence token prepended to each embedded method code
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim

        # Reuse the pretrained CodeT5 token embedding table directly.
        # Each token id indexes one row of this matrix.
        self.embed = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=freeze_embeddings,
            padding_idx=pad_id,
        )

        # CodeT5 embeddings may not match the LSTM hidden size. If they differ,
        # project embeddings down/up before passing them into the LSTMs.
        self.projection = (
            nn.Linear(embedding_dim, hidden_dim)
            if embedding_dim != hidden_dim else nn.Identity()
        )

        self.dropout = nn.Dropout(dropout)

        # The encoder reads the source code sequence and compresses it into the
        # final hidden/cell state tuple used to initialize the decoder.
        self.encoder = nn.LSTM(
            hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # The decoder consumes summary-prefix tokens and predicts the next token
        # at each position.
        self.decoder = nn.LSTM(
            hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Project decoder hidden states to logits over the tokenizer vocabulary.
        self.out = nn.Linear(hidden_dim, vocab_size)

    #embeds each sample in the batch
    def _embed(self, token_ids):
        # Convert token ids -> pretrained vectors -> optional projection ->
        # dropout-regularized sequence representation for the LSTM.
        embeddings = self.embed(token_ids)
        embeddings = self.projection(embeddings)
        return self.dropout(embeddings)

    def forward(self, src, tgt_inp):
        # Training path with teacher forcing:
        # 1. Encode the source code sequence.
        # 2. Feed the gt summary prefix into the decoder.
        # 3. Return logits for next-token prediction at each decoder step.
        _, hidden = self.encoder(self._embed(src))
        output, _ = self.decoder(self._embed(tgt_inp), hidden)
        return self.out(output)

    def generate(self, src, max_len=MAX_LEN):
        # Inference path:
        # 1. Encode the source code once.
        # 2. Start the decoder with the summary start token.
        # 3. Feed each prediction back into the decoder until EOS or max_len.
        _, hidden = self.encoder(self._embed(src))

        input_tok = torch.full(
            (src.size(0), 1),
            self.sos_id,
            dtype=torch.long,
            device=src.device,
        )

        generated = [[] for _ in range(src.size(0))]
        finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)

        for _ in range(max_len):
            output, hidden = self.decoder(self._embed(input_tok), hidden)
            logits = self.out(output[:, -1:, :])
            pred = logits.argmax(dim=-1)

            for i in range(src.size(0)):
                if finished[i]:
                    continue

                token_id = pred[i, 0].item()
                if token_id == self.eos_id:
                    finished[i] = True
                else:
                    generated[i].append(token_id)

            if finished.all():
                break

            input_tok = pred

        return generated


# Build the model with pretrained CodeT5 embeddings and standard training utilities.
model = Model(
    embedding_matrix=train_metadata["embedding_matrix"],
    pad_id=pad_id,
    eos_id=eos_id,
    sos_id=sos_id,
).to(DEVICE)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=0.001
)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

# Report model size so it is easy to compare frozen vs trainable parameter counts.
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,} total, {trainable_params:,} trainable")


Model parameters: 35,204,708 total, 35,204,708 trainable


## Computing BLEU-1 on Validation Set

In [19]:
def compute_bleu1(model, val_loader, tokenizer, bleu_metric):
    #switch to eval mode so dropout is disabled and validation is determinisitc
    model.eval()
    predictions = []
    references = []

    #validation does not need gradients becasuse we only generate text and score it
    with torch.no_grad():
        for src, tgt in val_loader:
            #move tensors onto target device
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)

            for i in range(src.size(0)):
                # Generate one predicted summary from the current model snapshot
                pred_ids = model.generate(src[i:i+1])[0]

                #Decode predicted token ids back into text, dropping special tokens
                #Like padding/end-of-sequence markers before BLEU scoring
                pred_text = tokenizer.decode(
                    pred_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )

                # Remove batch padding from the reference ids, then decode the gt summary with same tokenizer sttings used for pred.
                ref_ids = [token_id for token_id in tgt[i].tolist() if token_id != model.pad_id]
                ref_text = tokenizer.decode(
                    ref_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )

                #SacreBLEU expects system outputs as list[str] and references as list[list[str]]
                #Wrapping references once creats the single-reference corpus format it expects
                predictions.append(pred_text)
                references.append(ref_text)
    
    # Normalize SacreBLEU's native 0-100 score onto a 0-1 scale for logging
    # and checkpoint selection.
    return bleu_metric.corpus_score(predictions, [references]).score / 100.0

## Perform Training, Validate Every 500 Steps using BLEU-1, Save Best Model Checkpoint

In [20]:
# Track the best validation BLEU-1 and the number of consecutive misses.
best_bleu1 = 0
patience = 0

# Save the best checkpoint under a stable filename in the checkpoints directory.
save_dir = DEFAULT_DATA_DIR / "lstm_checkpoints"
os.makedirs(save_dir, exist_ok=True)
save_name = f"lstm_codet5_best"

# Precompute loop sizes so tqdm can show global training progress.
steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch

step = 0
running_loss = 0
log_steps = 50

# Train for multiple epochs, periodically evaluate on validation BLEU-1, and
# stop early once the score has stalled for three validations in a row.
progress = tqdm(total=total_steps, desc="Training")
for epoch in range(NUM_EPOCHS):
    model.train()
    for src, tgt in train_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        # Teacher forcing: the decoder consumes the gt summary prefix...
        output = model(src, tgt[:, :-1])

        # trained to predict the next summary token at each step.
        loss = criterion(output.reshape(-1, output.size(-1)),
                        tgt[:, 1:].reshape(-1))

        # Standard optimization step with gradient clipping for stability.
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        #one step is train on one sample from one batch
        step += 1
        running_loss += loss.item()

        if step % log_steps == 0:
            avg_loss = running_loss / log_steps
            progress.set_postfix({
                'loss': f'{avg_loss:.3f}',
                'epoch': f'{step/steps_per_epoch:.2f}'
            })
            running_loss = 0

        progress.update(1)

        # Evaluate the current snapshot every EVAL_STEPS and checkpoint the
        # model whenever validation BLEU-1 improves.
        if step % EVAL_STEPS == 0:
            bleu1 = compute_bleu1(model, val_loader, tokenizer, bleu_metric)
            tqdm.write(
                f"Step {step}/{total_steps} | BLEU-1 = {bleu1:.4f} | epoch {step/steps_per_epoch:.2f}",
                end="",
            )

            if bleu1 > best_bleu1:
                best_bleu1 = bleu1
                patience = 0
                torch.save(model.state_dict(), f'{save_dir}/{save_name}.pt')
                tqdm.write(" | Saved")
            else:
                patience += 1
                tqdm.write(f" | Patience {patience}/3")
                if patience >= 3:
                    # Break out once validation has failed to improve for
                    # three consecutive evaluations.
                    tqdm.write(f"\nEarly stopping at step {step}! Best BLEU-1: {best_bleu1:.4f}")
                    progress.close()
                    break

            model.train()
    else:
        continue
    break

progress.close()
print(f"\nBest BLEU-1: {best_bleu1:.4f}")


Training:  50%|██████████████████████████████████████████                                          | 2/4 [01:27<01:27, 43.91s/it]

Training:  50%|██████████████████████████████████████████                                          | 2/4 [00:00<00:00,  5.60it/s]
                                                                                                                              
Training:  50%|██████████████████████████████████████████                                          | 2/4 [00:00<00:00,  5.60it/s]
                                                                                                                              
Training:  75%|███████████████████████████████████████████████████████████████                     | 3/4 [00:00<00:00,  3.82it/s]

Step 2/4 | BLEU-1 = 0.0000 | epoch 0.50 | Patience 1/3



Training: 100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.41it/s]
                                                                                                                              
Training: 100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  4.41it/s]
                                                                                                                              
Training: 100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.16it/s]

Step 4/4 | BLEU-1 = 0.0458 | epoch 1.00 | Saved

Best BLEU-1: 0.0458


# Evaluate Model on Given Test Set and Compute Metrics 

## Import Dependencies for Evaluation and Set Up Constants

In [45]:
import argparse
import ast
import json
import os
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
import nltk
from nltk.translate import meteor_score as meteor_module
import bert_score

DEFAULT_CACHE_DIR = DEFAULT_DATA_DIR / ".runtime_cache"
# Keep matplotlib / Hugging Face cache writes inside the repo's writable data area.
os.environ.setdefault("MPLCONFIGDIR", str(DEFAULT_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(DEFAULT_CACHE_DIR))
LSTM_CHECKPOINT = DEFAULT_DATA_DIR / "lstm_checkpoints" / "lstm_codet5_best.pt"
SIDE_CHECKPOINT = DEFAULT_DATA_DIR / "side_checkpoint" / "141205"
DEFAULT_TEST_CSV = DEFAULT_DATA_DIR / "test_data" / "test_dataset_tokenized.csv"
DEFAULT_TRAIN_CODE_PT = DEFAULT_DATA_DIR / embeddings_dir / "train_code.pt"
DEFAULT_RESULTS_DIR = DEFAULT_DATA_DIR / "test_data" / "results"
DEFAULT_OUTPUT_JSON = DEFAULT_RESULTS_DIR / "lstm_test_predictions.json"
DEFAULT_METRICS_JSON = DEFAULT_RESULTS_DIR / "lstm_test_metrics.json"
DEFAULT_MAX_GEN_LEN = 100
DEFAULT_BATCH_SIZE = 16
DEFAULT_SIDE_BATCH_SIZE = 16

## Resolve Checkpoint and Training Metadata and Rebuild the Tokenizer and LSTM

Note all generated summaries for test set methods are stored in assignment_2/data/test_data/results/lstm_test_predictions.json. 

In [35]:
def load_training_metadata(train_code_pt: Path) -> dict:
    # Reuse the saved embedding matrix and tokenizer metadata from training.
    metadata = torch.load(train_code_pt, map_location="cpu")
    required_keys = {
        "embedding_matrix",
        "tokenizer_name",
        "pad_token_id",
        "eos_token_id",
        "vocab_size",
        "embedding_dim",
    }
    missing_keys = required_keys - set(metadata.keys())
    if missing_keys:
        raise ValueError(
            f"Training metadata is missing required keys: {sorted(missing_keys)}"
        )
    return metadata

def normalize_text(text: str) -> str:
    # Collapse repeated whitespace so predictions and references are stored cleanly.
    return " ".join(str(text).split())

def parse_id_list(raw_value: str) -> list[int]:
    # The CSV stores token ids as Python-list strings, so parse them back into ints.
    parsed = ast.literal_eval(raw_value)
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a list of token ids, got: {type(parsed).__name__}")
    return [int(token_id) for token_id in parsed]

def load_test_rows(test_csv: Path) -> list[dict]:
    # Load the tokenized CSV test split and keep only the fields needed for
    # generation, decoding, and metrics.
    df = pd.read_csv(test_csv)
    required_columns = {"code", "summary", "code_ids", "summary_ids"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Test CSV is missing required columns: {sorted(missing_columns)}"
        )

    rows = []
    for row_idx, record in enumerate(df.to_dict(orient="records")):
        rows.append(
            {
                "example_id": row_idx,
                "code": normalize_text(record["code"]),
                "summary": normalize_text(record["summary"]),
                "code_ids": parse_id_list(record["code_ids"]),
                "summary_ids": parse_id_list(record["summary_ids"]),
            }
        )
    return rows

def build_model(metadata: dict, checkpoint_path: Path, device: torch.device) -> Model:
    # Reconstruct the LSTM with the same special-token ids and embedding table
    # used in training, then load the saved weights for evaluation.
    pad_id = int(metadata["pad_token_id"])
    eos_id = int(metadata["eos_token_id"])
    # Training reused EOS as the decoder start token, so keep that convention.
    sos_id = eos_id

    model = Model(
        embedding_matrix=metadata["embedding_matrix"],
        pad_id=pad_id,
        eos_id=eos_id,
        sos_id=sos_id,
    )
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

device = torch.device(DEVICE)

checkpoint_path = LSTM_CHECKPOINT
metadata = load_training_metadata(DEFAULT_TRAIN_CODE_PT)
rows = load_test_rows(DEFAULT_TEST_CSV)

tokenizer = AutoTokenizer.from_pretrained(
    metadata["tokenizer_name"],
    local_files_only=True,
)
model = build_model(metadata, checkpoint_path, device)

## Generate Summaries for The Test Set

In [42]:
def strip_special_tokens(token_ids: list[int], pad_id: int, eos_id: int, sos_id: int) -> list[int]:
    # Remove control tokens before decoding summaries into plain text.
    return [
        token_id for token_id in token_ids
        if token_id not in {pad_id, eos_id, sos_id}
    ]

class TestCodeDataset(Dataset):
    # Wrap the tokenized test CSV rows so the DataLoader can batch code ids while
    # still preserving the original row metadata needed for decoding and output.
    def __init__(self, rows: list[dict], pad_id: int):
        self.rows = rows
        self.pad_id = pad_id

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        # Convert one code-id sequence into a tensor. The matching metadata row
        # is returned alongside it so evaluation can reconstruct outputs later.
        row = self.rows[idx]
        src = torch.tensor(row["code_ids"], dtype=torch.long)
        return src, row

def make_test_collate_fn(pad_id: int):
    # Pad a batch of variable-length code sequences into one rectangular tensor.
    def collate_fn(batch):
        src_batch, rows = zip(*batch)
        src_batch = pad_sequence(src_batch, batch_first=True, padding_value=pad_id)
        return src_batch, list(rows)

    return collate_fn

def generate_predictions(
    model: Model,
    tokenizer,
    rows: list[dict],
    batch_size: int,
    max_gen_len: int,
    device: torch.device,
) -> list[dict]:
    # Batch the test methods, generate one summary per method, then decode both
    # predictions and references back into plain text for metric computation.
    dataset = TestCodeDataset(rows, pad_id=model.pad_id)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_test_collate_fn(model.pad_id),
    )

    predictions = []
    with torch.no_grad():
        for src_batch, batch_rows in tqdm(dataloader, desc="Generating summaries"):
            # Move one padded batch of method token ids to the target device.
            src_batch = src_batch.to(device)
            generated_batch = model.generate(src_batch, max_len=max_gen_len)

            for row, pred_ids in zip(batch_rows, generated_batch):
                # Decode the generated token ids into a summary string.
                pred_text = tokenizer.decode(
                    pred_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )
                # Decode the ground-truth summary ids using the same tokenizer.
                ref_text = tokenizer.decode(
                    strip_special_tokens(
                        row["summary_ids"],
                        pad_id=model.pad_id,
                        eos_id=model.eos_id,
                        sos_id=model.sos_id,
                    ),
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )

                predictions.append(
                    {
                        "example_id": row["example_id"],
                        # Store the original method so outputs can be inspected later.
                        "code": row["code"],
                        "generated_summary": normalize_text(pred_text),
                        # Fall back to the raw CSV summary text if decoded ids become empty.
                        "reference_summary": normalize_text(ref_text or row["summary"]),
                    }
                )

    return predictions

def save_json(payload: dict | list, output_path: Path):
    # Materialize JSON outputs and create parent directories on demand.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)

triplets = generate_predictions(
    model=model,
    tokenizer=tokenizer,
    rows=rows,
    batch_size=DEFAULT_BATCH_SIZE,
    max_gen_len=DEFAULT_MAX_GEN_LEN,
    device=device,
)
save_json(triplets, DEFAULT_OUTPUT_JSON)
print("sample triplets saved to:", DEFAULT_OUTPUT_JSON)

Generating summaries: 100%|████████████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  2.93it/s]

sample triplets saved to /Users/sambennett/Desktop/CSCI555/assignment_2/data/test_data/results/lstm_test_predictions.json


## Compute and Save Metrics

In [50]:
def compute_bleu_scores(predictions: list[str], references: list[str]) -> dict:
    # Compute BLEU-1 through BLEU-4 with SacreBLEU's standard corpus scorer.
    bleu_scores = {}
    for ngram_order in range(1, 5):
        metric = sacrebleu.metrics.BLEU(max_ngram_order=ngram_order)
        score = metric.corpus_score(predictions, [references]).score
        bleu_scores[f"bleu_{ngram_order}"] = float(score)
    return bleu_scores

def compute_meteor(predictions: list[str], references: list[str]) -> dict:
    # METEOR is averaged over examples after tokenizing summaries on whitespace.
    scores = [
        meteor_module.meteor_score([reference.split()], prediction.split())
        for prediction, reference in zip(predictions, references)
    ]
    return {
        "meteor": float(sum(scores) / len(scores)) if scores else 0.0,
    }

def compute_bertscore(
    predictions: list[str],
    references: list[str],
    batch_size: int,
    device: torch.device,
) -> dict:
    # BERTScore returns one precision/recall/F1 triple per example; report the
    # dataset-level mean of each component.

    precision, recall, f1 = bert_score.score(
        predictions,
        references,
        lang="en",
        batch_size=batch_size,
        device=str(device),
        rescale_with_baseline=True,
    )
    return {
        "bertscore_precision": float(precision.mean().item()),
        "bertscore_recall": float(recall.mean().item()),
        "bertscore_f1": float(f1.mean().item()),
    }

def mean_pooling(model_output, attention_mask: torch.Tensor) -> torch.Tensor:
    # Average token embeddings over only the non-padding positions.
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
        input_mask_expanded.sum(dim=1), min=1e-9
    )

def compute_side(
    code_texts: list[str],
    generated_summaries: list[str],
    side_checkpoint: Path | None,
    batch_size: int,
    device: torch.device,
    local_files_only: bool,
) -> dict:
    # SIDE scores how well a generated summary semantically aligns with the code
    # itself, independent of the reference summary, using the authors' encoder.
    if side_checkpoint is None:
        raise RuntimeError(
            "SIDE requires a local checkpoint from the original implementation. "
            "Pass it with --side-checkpoint."
        )
    if not side_checkpoint.exists():
        raise FileNotFoundError(f"SIDE checkpoint not found: {side_checkpoint}")

    tokenizer = AutoTokenizer.from_pretrained(
        side_checkpoint,
        local_files_only=local_files_only,
    )
    model = AutoModel.from_pretrained(
        side_checkpoint,
        local_files_only=local_files_only,
    ).to(device)
    model.eval()

    similarities = []
    with torch.no_grad():
        for start_idx in tqdm(range(0, len(code_texts), batch_size), desc="Computing SIDE"):
            # Process aligned code / generated-summary pairs in batches.
            batch_codes = code_texts[start_idx:start_idx + batch_size]
            batch_summaries = generated_summaries[start_idx:start_idx + batch_size]

            # Encode code snippets and summaries separately using the SIDE encoder.
            encoded_codes = tokenizer(
                batch_codes,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(device)
            encoded_summaries = tokenizer(
                batch_summaries,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(device)

            code_output = model(**encoded_codes)
            summary_output = model(**encoded_summaries)

            # Convert token-level outputs into one normalized vector per input.
            code_embeddings = mean_pooling(code_output, encoded_codes["attention_mask"])
            summary_embeddings = mean_pooling(summary_output, encoded_summaries["attention_mask"])

            code_embeddings = F.normalize(code_embeddings, p=2, dim=1)
            summary_embeddings = F.normalize(summary_embeddings, p=2, dim=1)

            # Compare each code embedding only with its aligned summary embedding.
            batch_similarities = F.cosine_similarity(code_embeddings, summary_embeddings, dim=1)
            similarities.extend(batch_similarities.cpu().tolist())

    return {
        # Report the test-set mean SIDE score across all aligned examples.
        "side": float(sum(similarities) / len(similarities)) if similarities else 0.0,
    }

predictions = [row["generated_summary"] for row in triplets]
references = [row["reference_summary"] for row in triplets]
code_texts = [row["code"] for row in triplets]

# Store corpus-level metric outputs in one JSON object for easy reporting.
metrics = {
    "checkpoint_path": str(checkpoint_path),
    "num_examples": len(triplets),
}
metrics.update(compute_bleu_scores(predictions, references))
metrics.update(compute_meteor(predictions, references))
metrics.update(
    compute_bertscore(
        predictions=predictions,
        references=references,
        batch_size=DEFAULT_BATCH_SIZE,
        device=device,
    )
)

metrics.update(
    compute_side(
        code_texts=code_texts,
        generated_summaries=predictions,
        side_checkpoint=SIDE_CHECKPOINT,
        batch_size=DEFAULT_SIDE_BATCH_SIZE,
        device=device,
        local_files_only=True,
    )
)

save_json(metrics, DEFAULT_METRICS_JSON)
print(json.dumps(metrics, indent=2))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Computing SIDE: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:08<00:00,  1.19s/it]

{
  "checkpoint_path": "/Users/sambennett/Desktop/CSCI555/assignment_2/data/lstm_checkpoints/lstm_codet5_best.pt",
  "num_examples": 99,
  "bleu_1": 1.8585858585858586,
  "bleu_2": 0.09737357649578737,
  "bleu_3": 0.029017076897571593,
  "bleu_4": 0.013354142816961281,
  "meteor": 0.0,
  "bertscore_precision": -0.503364622592926,
  "bertscore_recall": -0.1906958818435669,
  "bertscore_f1": -0.350569486618042,
  "side": 0.22341990802963876
}
